# SemEval 2027 Task 9 - Subtask 1: Framing in Text
## Thực nghiệm Đột phá: Hướng 03 (Advanced Large Multilingual Backbone) + Hướng 04 (Context-Enriched Input) + Hướng 05 (Leakage-Prevention Group Split) + Hướng 01 (Per-Class Thresholds)

Notebook này nâng cấp toàn diện từ `dev2_h1_h4_h5.ipynb`, tích hợp thêm **Hướng 03 (Khám phá Backbone Tiên tiến & Mạnh hơn)**:
1. **Hướng 03 — Backbone Đa ngôn ngữ Cỡ lớn (Large Model):**
   * Sử dụng **`FacebookAI/xlm-roberta-large`** (560M tham số, 24 transformer layers, hidden dimension 1024, 16 attention heads) — gấp 3.5 lần sức chứa tri thức so với bản `base`, tăng cường biểu diễn ngữ nghĩa và khả năng Zero-shot Cross-lingual Transfer sang các ngôn ngữ unseen/low-resource (KA, GR, ES).
   * Hỗ trợ chuyển đổi nhanh sang **`microsoft/mdeberta-v3-base`** (Disentangled Attention) hoặc các mô hình tiên tiến khác.
2. **Kỹ thuật Huấn luyện Tối ưu cho Large Backbone:**
   * **Gradient Checkpointing:** Giảm tải VRAM để huấn luyện mô hình Large ổn định.
   * **Learning Rate Thích ứng:** `1.2e-5` cùng Warmup và Weight Decay chống phân kỳ.
3. **Kế thừa Toàn vẹn Hệ sinh thái Thử nghiệm Chuẩn mực:**
   * **Hướng 04:** Ngữ cảnh Tiêu đề bài báo (`<s> Title </s></s> Paragraph </s>`) với Audited Token Budget (Max 306 tokens).
   * **Hướng 05:** Stratified Group Split theo tài liệu (`doc_id`), cam kết 100% Zero Data Leakage.
   * **Hướng 01:** Coordinate Search 14 ngưỡng per-class trên tập Validation.
   * **Scorer Chính thức:** 15 lớp (14 frames + nhãn `None`, `zero_division=1`).


## 1. Tự động Giải nén Dữ liệu vào Máy ảo (Kaggle Workspace)


In [1]:
# 1. TỰ ĐỘNG GIẢI NÉN DỮ LIỆU VÀO MÁY ẢO 
# Đường dẫn thư mục đích trên máy ảo, cần giải nén để tránh load nhiều file .txt 
import os
from pathlib import Path

WORK_DATA_DIR = "/kaggle/working/data"

if not os.path.exists(WORK_DATA_DIR) and os.path.exists("/kaggle/working"):
    print("[1/2] Đang copy file dữ liệu .bin ...")
    # Đường dẫn file zip data trên Kaggle (Dataset: quctriunguyn/datazip)
    !cp /kaggle/input/datasets/quctriunguyn/datazip/data-second-batch.zip.bin /kaggle/working/data.zip
    print("[2/2] Đang giải nén thẳng vào ổ cứng máy ảo (sẽ mất vài giây)...")
    !unzip -q /kaggle/working/data.zip -d /kaggle/working/


[1/2] Đang copy file dữ liệu .bin ...
[2/2] Đang giải nén thẳng vào ổ cứng máy ảo (sẽ mất vài giây)...


## 2. Cài đặt các thư viện cần thiết


In [2]:
# Cài đặt các thư viện phụ thuộc (Bỏ comment nếu môi trường chưa có sẵn)
#!pip install -q transformers datasets accelerate scikit-learn


## 3. Khai báo Thư viện & Cố định Seed


In [3]:
import os
import gc
import csv
import json
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_recall_fscore_support, classification_report

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

# Kiểm tra thiết bị phần cứng GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị sử dụng: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Thiết lập Random Seed cố định
SEED = 42
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)

set_all_seeds(SEED)


Thiết bị sử dụng: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 94.97 GB


## 4. Cấu hình Đường dẫn Dự án & Siêu tham số (Hướng 03: Large Backbone)


In [4]:
# 2. CẤU HÌNH ĐƯỜNG DẪN DỰ ÁN
# A. Thư mục gốc dữ liệu (Trỏ thẳng vào thư mục vừa giải nén ở Cell 1)
DATA_DIR = Path("/kaggle/working/data")
if not DATA_DIR.exists() and Path("data").exists():
    DATA_DIR = Path("data")
elif not DATA_DIR.exists() and Path("/kaggle/input/datasets/quctriunguyn/dataset-task9/data").exists():
    DATA_DIR = Path("/kaggle/input/datasets/quctriunguyn/dataset-task9/data")

# B. Các gói dữ liệu con
TRAIN_DATA_PATH   = DATA_DIR / "initial-training-data/subtask1/SEMEVAL23_FRAMING_INITIAL_TRAIN-A"
TEST_DATA_PATH    = DATA_DIR / "initial-training-data/subtask1/SEMEVAL23_FRAMING_INITIAL_TRAIN-B"
SECOND_BATCH_PATH = DATA_DIR / "training-second-batch-oct"

# C. HƯỚNG 03: LỰA CHỌN BACKBONE MẠNH MẼ (LARGE MODEL)
# Ưu tiên: FacebookAI/xlm-roberta-large (hoặc đường dẫn offline nếu có)
MODEL_NAME = "/kaggle/input/models/aisuko/xlm-roberta-large/transformers/default/1"
OFFLINE_CANDIDATES = [
    "/kaggle/input/models/aisuko/xlm-roberta-large/transformers/default/1",
    "/kaggle/input/xlm-roberta-large",
    "/kaggle/input/datasets/quctriunguyn/my-xlm-roberta-offline/xlm-roberta-base-offline"
]
for p in OFFLINE_CANDIDATES:
    if Path(p).exists():
        MODEL_NAME = p
        break

# D. Thư mục đầu ra (Lưu Checkpoint, File TSV nộp bài)
OUTPUT_DIR = Path("/kaggle/working/output" if Path("/kaggle/working").exists() else "./output")
MODEL_SAVE_DIR = Path("/kaggle/working/model_large" if Path("/kaggle/working").exists() else "./model_large")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# CẤU HÌNH PROTOCOL THỰC NGHIỆM & TOKEN BUDGET AUDITED
# ==============================================================================
# Chọn Protocol:
#   - "benchmark_ab": Protocol 1 (Huấn luyện Gói A, Dò ngưỡng Val A, Kiểm nghiệm mù Test B)
#   - "full_train"  : Protocol 2 (Huấn luyện trên toàn bộ dữ liệu A + B + Second Batch)
EXPERIMENT_PROTOCOL = "benchmark_ab"

# Ngân sách Token Chuẩn (Audited Token Budget từ bs2023AB_da_train):
MAX_LENGTH = 306
PARAGRAPH_MAX_TOKENS = 254
TITLE_MAX_TOKENS = 48

# Siêu tham số huấn luyện tối ưu cho Backbone Cỡ lớn (Large)
IS_LARGE_MODEL = "large" in MODEL_NAME.lower()
LEARNING_RATE = 1.2e-5 if IS_LARGE_MODEL else 2e-5    # Large model yêu cầu LR nhỏ hơn để ổn định
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION = 2     # Effective batch size = 8 * 2 = 16
NUM_EPOCHS = 18
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 300            # Warmup steps
VAL_SIZE = 0.10               # 10% nhóm tài liệu làm Validation
INFERENCE_THRESHOLD = 0.50

# Trọng số bù đắp mất cân bằng nhãn (sqrt_cap4)
POS_WEIGHT_POWER = 0.5
POS_WEIGHT_MAX = 4.0

print(f"\nCấu hình thực nghiệm (Hướng 03 + 04 + 05 + 01):")
print(f"  • Thư mục dữ liệu : {DATA_DIR.resolve()}")
print(f"  • Model Backbone  : {MODEL_NAME}")
print(f"  • Loại Mô hình    : {'Large (560M params)' if IS_LARGE_MODEL else 'Base'}")
print(f"  • Learning Rate   : {LEARNING_RATE}")
print(f"  • Protocol        : {EXPERIMENT_PROTOCOL}")
print(f"  • Token Budget    : MAX={MAX_LENGTH} (Para={PARAGRAPH_MAX_TOKENS}, Title={TITLE_MAX_TOKENS})")
print(f"  • Epochs          : {NUM_EPOCHS} | Effective Batch Size: {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION}")



Cấu hình thực nghiệm (Hướng 03 + 04 + 05 + 01):
  • Thư mục dữ liệu : /kaggle/working/data
  • Model Backbone  : /kaggle/input/models/aisuko/xlm-roberta-large/transformers/default/1
  • Loại Mô hình    : Large (560M params)
  • Learning Rate   : 1.2e-05
  • Protocol        : benchmark_ab
  • Token Budget    : MAX=306 (Para=254, Title=48)
  • Epochs          : 18 | Effective Batch Size: 16


## 5. Danh mục 14 Khung Framing Chuẩn SemEval Task 9


In [5]:
# 14 Khung Diễn giải Chuẩn (Framing Categories) theo thứ tự bảng chữ cái
LABEL_COLUMNS = sorted([
    "Capacity_and_resources",
    "Crime_and_punishment",
    "Cultural_identity",
    "Economic",
    "External_regulation_and_reputation",
    "Fairness_and_equality",
    "Health_and_safety",
    "Legality_Constitutionality_and_jurisprudence",
    "Morality",
    "Policy_prescription_and_evaluation",
    "Political",
    "Public_opinion",
    "Quality_of_life",
    "Security_and_defense"
])
NUM_LABELS = len(LABEL_COLUMNS)
print(f"Tổng số nhãn chuẩn: {NUM_LABELS}")
for idx, lbl in enumerate(LABEL_COLUMNS, start=1):
    print(f"  {idx:2d}. {lbl}")


Tổng số nhãn chuẩn: 14
   1. Capacity_and_resources
   2. Crime_and_punishment
   3. Cultural_identity
   4. Economic
   5. External_regulation_and_reputation
   6. Fairness_and_equality
   7. Health_and_safety
   8. Legality_Constitutionality_and_jurisprudence
   9. Morality
  10. Policy_prescription_and_evaluation
  11. Political
  12. Public_opinion
  13. Quality_of_life
  14. Security_and_defense


## 6. Hướng 04: Tải Dữ liệu & Trích xuất Tiêu đề Bài báo (Context Enrichment)

Mỗi bài báo trong thư mục `raw-documents` tuân theo cấu trúc:
* **Dòng 1:** Tiêu đề bài báo (**Title**).
* **Dòng 2:** Dòng trống.
* **Dòng 3 trở đi:** Thân bài báo.

Trích xuất annotations từ `subtask-1-annotations-with-paragraphs.txt` và ghép nối chính xác với Tiêu đề bài báo.


In [6]:
def load_dataset_package(target_path: Path, package_name: str = "Dataset"):
    """Tải các đoạn văn bản kèm tiêu đề bài báo từ thư mục chỉ định."""
    if not target_path.exists():
        print(f"[Cảnh báo] Thư mục {target_path} không tồn tại!")
        return pd.DataFrame(), None

    raw_files = sorted(target_path.rglob("subtask-1-annotations-with-paragraphs.txt"))
    if not raw_files:
        # Fallback tìm kiếm linh hoạt nếu tên file thay đổi nhẹ
        raw_files = sorted(target_path.rglob("*annotation*paragraph*.txt"))

    if not raw_files:
        print(f"[Cảnh báo] Không tìm thấy file annotations trong {target_path}")
        return pd.DataFrame(), None

    print(f"--- Đang tải {package_name} ({len(raw_files)} annotation files) ---")

    # 1. Trích xuất Tiêu đề từ raw-documents tương ứng (Hướng 04)
    title_map = {}
    for annot_path in raw_files:
        raw_doc_dir = annot_path.parent / "raw-documents"
        if raw_doc_dir.exists():
            for txt_file in raw_doc_dir.glob("*.txt"):
                try:
                    with open(txt_file, "r", encoding="utf-8-sig", errors="replace") as f:
                        first_line = f.readline().strip()
                        if first_line:
                            title_map[txt_file.name] = first_line
                except Exception:
                    pass

    # 2. Đọc các đoạn văn và gán Title
    records = []
    skipped = 0
    matched_title = 0
    valid_labels_set = set(LABEL_COLUMNS)

    for path in raw_files:
        parent_name = path.parent.name
        lang = parent_name.split("_")[0].split("-")[0].upper()

        with path.open("r", encoding="utf-8-sig", errors="replace", newline="") as handle:
            reader = csv.reader(handle, delimiter="\t")
            for row in reader:
                if len(row) < 5:
                    skipped += 1
                    continue

                doc_id = row[0].strip()
                begin_span = row[1].strip()
                end_span = row[2].strip()
                paragraph = row[-1].strip()

                if not paragraph:
                    skipped += 1
                    continue

                raw_labels = [l.strip() for l in row[3:-1] if l.strip()]
                labels = [l for l in raw_labels if l in valid_labels_set]

                doc_title = title_map.get(doc_id, "")
                if doc_title:
                    matched_title += 1

                records.append({
                    "doc_id": doc_id,
                    "begin_span": begin_span,
                    "end_span": end_span,
                    "language": lang,
                    "title": doc_title,
                    "paragraph": paragraph,
                    "labels": labels,
                    "source_package": package_name
                })

    df = pd.DataFrame(records)
    if len(df) == 0:
        print(f"[Cảnh báo] Không trích xuất được bản ghi nào từ {package_name}!")
        return pd.DataFrame(), None

    print(f"=> {package_name}: {len(df):,} đoạn văn từ {df['doc_id'].nunique():,} bài báo.")
    print(f"=> Tỉ lệ có tiêu đề: {matched_title}/{len(df)} ({matched_title/max(len(df),1)*100:.2f}%)")

    # Mã hóa Multi-hot 14 nhãn
    mlb = MultiLabelBinarizer(classes=LABEL_COLUMNS)
    encoded = mlb.fit_transform(df["labels"])
    labels_df = pd.DataFrame(encoded, columns=LABEL_COLUMNS, dtype=np.int8)

    full_df = pd.concat([df.drop(columns=["labels"]), labels_df], axis=1)
    return full_df, mlb

# Tải dữ liệu tùy theo Protocol
if EXPERIMENT_PROTOCOL == "benchmark_ab" and TRAIN_DATA_PATH.exists() and TEST_DATA_PATH.exists():
    print("=== CHẾ ĐỘ PROTOCOL 1: BENCHMARK KHOA HỌC A -> B ===")
    dev_raw_df, mlb = load_dataset_package(TRAIN_DATA_PATH, package_name="Package_A")
    test_raw_df, _ = load_dataset_package(TEST_DATA_PATH, package_name="Package_B")
    has_test_b = len(test_raw_df) > 0
else:
    print("=== CHẾ ĐỘ PROTOCOL 2: FULL DATASET TRAINING ===")
    dev_raw_df, mlb = load_dataset_package(DATA_DIR, package_name="Full_Data")
    test_raw_df = pd.DataFrame()
    has_test_b = False

assert len(dev_raw_df) > 0 and "doc_id" in dev_raw_df.columns, (
    f"Lỗi: Không tìm thấy dữ liệu hợp lệ trong {DATA_DIR}! Hãy kiểm tra xem bước giải nén ở Cell 1 đã hoàn tất chưa."
)

print(f"\nVí dụ mẫu dữ liệu sau khi trích xuất Tiêu đề:")
print(f"Doc ID   : {dev_raw_df.iloc[0]['doc_id']} (Lang: {dev_raw_df.iloc[0]['language']})")
print(f"Title    : {dev_raw_df.iloc[0]['title']}")
print(f"Paragraph: {dev_raw_df.iloc[0]['paragraph'][:100]}...")
dev_raw_df.head(2)


=== CHẾ ĐỘ PROTOCOL 1: BENCHMARK KHOA HỌC A -> B ===
--- Đang tải Package_A (5 annotation files) ---
=> Package_A: 4,077 đoạn văn từ 1,076 bài báo.
=> Tỉ lệ có tiêu đề: 3997/4077 (98.04%)
--- Đang tải Package_B (9 annotation files) ---
=> Package_B: 1,720 đoạn văn từ 450 bài báo.
=> Tỉ lệ có tiêu đề: 1718/1720 (99.88%)

Ví dụ mẫu dữ liệu sau khi trích xuất Tiêu đề:
Doc ID   : DE_TRAIN_doc_1.txt (Lang: DE)
Title    : Russland findet weiterhin Abnehmer für sein Erdöl
Paragraph: Russland findet weiterhin Abnehmer für sein Erdöl...


,doc_id,begin_span,end_span,language,title,paragraph,source_package,Capacity_and_resources,Crime_and_punishment,Cultural_identity,...,External_regulation_and_reputation,Fairness_and_equality,Health_and_safety,Legality_Constitutionality_and_jurisprudence,Morality,Policy_prescription_and_evaluation,Political,Public_opinion,Quality_of_life,Security_and_defense
0,DE_TRAIN_doc_1.txt,0,49,DE,Russland findet weiterhin Abnehmer für sein Erdöl,Russland findet weiterhin Abnehmer für sein Erdöl,Package_A,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,DE_TRAIN_doc_1.txt,51,100,DE,Russland findet weiterhin Abnehmer für sein Erdöl,Westliche Öl-Sanktionen erweist sich als Eigentor,Package_A,0,0,0,...,1,0,0,0,0,0,0,0,0,0


## 7. Hướng 05: Phân chia Tập Train / Val theo Tài liệu (Zero Data Leakage)

Để ngăn ngừa triệt để hiện tượng mô hình học thuộc lòng phong cách viết hay thông tin đặc thù của từng bài báo, thuật toán **Stratified Group Split**:
1. Gom nhóm toàn bộ đoạn văn theo `doc_id`.
2. Phân bổ các bài báo chứa nhãn hiếm vào tập Val trước để cân bằng phân bố.
3. **Đảm bảo độ rò rỉ bằng 0 (`overlap = 0`)**: Một bài báo chỉ nằm ở Train HOẶC Val.


In [7]:
def stratified_group_train_val_split(
    df: pd.DataFrame,
    group_col: str = "doc_id",
    label_cols: list[str] = LABEL_COLUMNS,
    val_size: float = 0.10,
    random_state: int = 42
):
    assert group_col in df.columns, f"Lỗi: Cột '{group_col}' không tồn tại trong dữ liệu! Các cột hiện có: {list(df.columns)}"
    np.random.seed(random_state)

    # 1. Thống kê nhãn theo từng bài báo (doc_id)
    doc_label_counts = df.groupby(group_col)[label_cols].sum()
    unique_docs = doc_label_counts.index.values
    np.random.shuffle(unique_docs)

    total_samples = len(df)
    target_val_samples = int(total_samples * val_size)
    total_label_counts = df[label_cols].sum()
    target_val_labels = (total_label_counts * val_size).to_dict()

    current_val_docs = set()
    current_val_labels = defaultdict(int)
    current_val_sample_count = 0

    # 2. Xếp thứ tự ưu tiên: bài báo chứa nhãn hiếm được duyệt trước
    rare_weights = 1.0 / (total_label_counts + 1e-5)
    doc_rarity = {doc: float((doc_label_counts.loc[doc] * rare_weights).sum()) for doc in unique_docs}
    sorted_docs = sorted(unique_docs, key=lambda d: doc_rarity[d], reverse=True)

    # 3. Phân bổ bài báo vào tập Validation
    for doc in sorted_docs:
        doc_rows = len(df[df[group_col] == doc])
        if current_val_sample_count + doc_rows > target_val_samples * 1.15 and len(current_val_docs) > 0:
            continue

        doc_counts = doc_label_counts.loc[doc]
        should_add = any(int(doc_counts[lbl]) > 0 and current_val_labels[lbl] < target_val_labels[lbl] for lbl in label_cols)

        if should_add or (current_val_sample_count < target_val_samples * 0.85):
            current_val_docs.add(doc)
            current_val_sample_count += doc_rows
            for lbl in label_cols:
                current_val_labels[lbl] += int(doc_counts[lbl])

        if current_val_sample_count >= target_val_samples:
            break

    # 4. Trích xuất chỉ số hàng (indices)
    val_mask = df[group_col].isin(current_val_docs)
    val_indices = df[val_mask].index.values
    train_indices = df[~val_mask].index.values

    train_docs = set(df.iloc[train_indices][group_col])
    val_docs = set(df.iloc[val_indices][group_col])
    leakage = train_docs.intersection(val_docs)

    print(f"=== KẾT QUẢ PHÂN CHIA GROUP SPLIT THEO TÀI LIỆU ===")
    print(f"Tổng số bài báo (doc_id): {len(unique_docs):,}")
    print(f"  • Train bài báo : {len(train_docs):,}")
    print(f"  • Val bài báo   : {len(val_docs):,}")
    print(f"Tổng số đoạn văn : Train = {len(train_indices):,} ({len(train_indices)/total_samples*100:.1f}%) | Val = {len(val_indices):,} ({len(val_indices)/total_samples*100:.1f}%)")

    # Kiểm tra rò rỉ dữ liệu
    assert len(leakage) == 0, f"LỖI: Phát hiện rò rỉ {len(leakage)} tài liệu trùng lặp!"
    print(f"=> XÁC NHẬN: RÒ RỈ DỮ LIỆU BẰNG 0 (ZERO DATA LEAKAGE)!\n")

    return train_indices, val_indices

train_idx, val_idx = stratified_group_train_val_split(
    dev_raw_df, group_col="doc_id", label_cols=LABEL_COLUMNS, val_size=VAL_SIZE, random_state=SEED
)

train_df = dev_raw_df.iloc[train_idx].reset_index(drop=True)
val_df = dev_raw_df.iloc[val_idx].reset_index(drop=True)

print("Phân bố số lượng nhãn trong tập Validation:")
val_stats = pd.DataFrame({
    "Tổng số trong Val": val_df[LABEL_COLUMNS].sum(),
    "Tỉ lệ trong Val (%)": (val_df[LABEL_COLUMNS].sum() / dev_raw_df[LABEL_COLUMNS].sum() * 100).round(1)
})
display(val_stats)


=== KẾT QUẢ PHÂN CHIA GROUP SPLIT THEO TÀI LIỆU ===
Tổng số bài báo (doc_id): 1,076
  • Train bài báo : 988
  • Val bài báo   : 88
Tổng số đoạn văn : Train = 3,731 (91.5%) | Val = 346 (8.5%)
=> XÁC NHẬN: RÒ RỈ DỮ LIỆU BẰNG 0 (ZERO DATA LEAKAGE)!

Phân bố số lượng nhãn trong tập Validation:


,Tổng số trong Val,Tỉ lệ trong Val (%)
Capacity_and_resources,30,7.2
Crime_and_punishment,24,9.8
Cultural_identity,14,8.1
Economic,48,6.9
External_regulation_and_reputation,35,8.0
Fairness_and_equality,17,10.6
Health_and_safety,52,8.7
Legality_Constitutionality_and_jurisprudence,31,11.4
Morality,24,9.6
Policy_prescription_and_evaluation,26,7.6


## 8. Xây dựng PyTorch Dataset & Tokenizer với Ngân sách Token Audited

Áp dụng chuẩn Token Budget từ `bs2023AB_da_train.ipynb`:
* Tokenize riêng `title` (tối đa 48 tokens) và `paragraph` (tối đa 254 tokens).
* Ghép chuỗi theo cấu trúc đặc tả của XLM-RoBERTa:
  $$\text{input\_ids} = [\text{CLS}] + \text{title\_ids} + [\text{SEP}, \text{SEP}] + \text{para\_ids} + [\text{SEP}]$$
* Đảm bảo tổng chiều dài luôn $\le 306$ tokens và **không bao giờ để tiêu đề dài chèn ép làm mất nội dung đoạn văn**.


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AuditedContextParagraphDataset(Dataset):
    """Dataset mã hóa Tiêu đề và Đoạn văn theo ngân sách token đã audit (Hướng 04)."""
    def __init__(
        self,
        dataframe: pd.DataFrame,
        tokenizer,
        label_columns: list[str] = LABEL_COLUMNS,
        max_length: int = MAX_LENGTH,
        paragraph_max_tokens: int = PARAGRAPH_MAX_TOKENS,
        title_max_tokens: int = TITLE_MAX_TOKENS,
        is_test: bool = False
    ):
        self.paragraphs = dataframe["paragraph"].fillna("").astype(str).tolist()
        self.titles = dataframe["title"].fillna("").astype(str).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.paragraph_max_tokens = paragraph_max_tokens
        self.title_max_tokens = title_max_tokens
        self.is_test = is_test

        self.cls_id = tokenizer.cls_token_id
        self.sep_id = tokenizer.sep_token_id

        if not self.is_test and all(col in dataframe.columns for col in label_columns):
            self.labels = dataframe[label_columns].values.astype(np.float32)
        else:
            self.labels = None

    def __len__(self):
        return len(self.paragraphs)

    def __getitem__(self, idx):
        para = self.paragraphs[idx]
        title = self.titles[idx]

        # Tokenize riêng với budget cố định (Hướng 04)
        p_ids = self.tokenizer.encode(para, add_special_tokens=False, truncation=True, max_length=self.paragraph_max_tokens)
        t_ids = self.tokenizer.encode(title, add_special_tokens=False, truncation=True, max_length=self.title_max_tokens) if title else []

        # Ghép cặp câu chuẩn XLM-RoBERTa: <s> Title </s></s> Paragraph </s>
        if t_ids:
            input_ids = [self.cls_id] + t_ids + [self.sep_id, self.sep_id] + p_ids + [self.sep_id]
        else:
            input_ids = [self.cls_id] + p_ids + [self.sep_id]

        item = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor([1] * len(input_ids), dtype=torch.long),
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)

        return item

class DynamicPaddingDataCollator:
    """Collator đệm động theo chuỗi dài nhất trong mini-batch để tiết kiệm VRAM."""
    def __init__(self, tokenizer, pad_to_multiple_of: int = 8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        has_labels = "labels" in features[0]
        if has_labels:
            labels = torch.stack([f["labels"] for f in features])

        inputs = [{k: v for k, v in f.items() if k != "labels"} for f in features]
        batch = self.tokenizer.pad(
            inputs,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt"
        )
        if has_labels:
            batch["labels"] = labels
        return batch

train_dataset = AuditedContextParagraphDataset(train_df, tokenizer, LABEL_COLUMNS)
val_dataset = AuditedContextParagraphDataset(val_df, tokenizer, LABEL_COLUMNS)
collator = DynamicPaddingDataCollator(tokenizer)

print(f"Ví dụ tensor đầu vào của mẫu 0:")
sample_item = train_dataset[0]
print(f"  • input_ids shape      : {sample_item['input_ids'].shape} (Tokens: {len(sample_item['input_ids'])})")
print(f"  • attention_mask shape : {sample_item['attention_mask'].shape}")
print(f"  • labels               : {sample_item['labels'].numpy()}")


Ví dụ tensor đầu vào của mẫu 0:
  • input_ids shape      : torch.Size([54]) (Tokens: 54)
  • attention_mask shape : torch.Size([54])
  • labels               : [1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]


## 9. Xử lý Mất cân bằng Nhãn & WeightedMultilabelTrainer

Do 14 nhãn framing có tỉ lệ mất cân bằng lớn (nhãn `Political` xuất hiện nhiều, trong khi `Fairness_and_equality`, `Cultural_identity` xuất hiện rất ít), ta tính trọng số `pos_weight` theo công thức căn bậc hai tỉ lệ âm/dương (`sqrt_cap4`):
$$\text{pos\_weight}_c = \min\left(\left(\frac{N - N_c^+}{N_c^+}\right)^{0.5}, 4.0\right)$$


In [9]:
def compute_pos_weight(dataframe, label_columns, power=POS_WEIGHT_POWER, max_weight=POS_WEIGHT_MAX):
    pos_counts = dataframe[label_columns].sum(axis=0).to_numpy(dtype=np.float32)
    neg_counts = len(dataframe) - pos_counts
    raw_ratio = neg_counts / np.maximum(pos_counts, 1.0)
    pos_weights = np.clip(np.power(raw_ratio, power), 1.0, max_weight)
    return torch.tensor(pos_weights, dtype=torch.float32)

pos_weight = compute_pos_weight(train_df, LABEL_COLUMNS)
print("Trọng số pos_weight trên tập Train:")
for col, w in zip(LABEL_COLUMNS, pos_weight):
    print(f"  {col:45s}: {w:.4f}")

class WeightedMultilabelTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weight = self.pos_weight.to(logits.device) if self.pos_weight is not None else None
        loss = F.binary_cross_entropy_with_logits(logits, labels.float(), pos_weight=weight)
        return (loss, outputs) if return_outputs else loss


Trọng số pos_weight trên tập Train:
  Capacity_and_resources                       : 2.9353
  Crime_and_punishment                         : 3.9757
  Cultural_identity                            : 4.0000
  Economic                                     : 2.1812
  External_regulation_and_reputation           : 2.8777
  Fairness_and_equality                        : 4.0000
  Health_and_safety                            : 2.4178
  Legality_Constitutionality_and_jurisprudence : 3.7970
  Morality                                     : 3.9289
  Policy_prescription_and_evaluation           : 3.2761
  Political                                    : 2.2866
  Public_opinion                               : 4.0000
  Quality_of_life                              : 3.6999
  Security_and_defense                         : 2.2336


## 10. Định nghĩa Thước đo Đánh giá & Scorer Chính thức SemEval 2027

Hệ thống tính toán đồng thời 4 chỉ số:
1. **Official Micro-F1:** Thước đo xếp hạng chính thức của cuộc thi SemEval 2027 Subtask 1 (đánh giá trên 15 lớp: 14 frames + `None`, `zero_division=1`).
2. **Official Macro-F1:** Macro-F1 trên 15 lớp.
3. **Frames14 Micro-F1:** Micro-F1 thuần túy trên 14 khung diễn giải.
4. **Frames14 Macro-F1:** Thước đo so sánh cốt lõi giữa các fold và mô hình (`zero_division=0`).


In [10]:
def compute_official_semeval_metrics(y_true_binary: np.ndarray, y_pred_binary: np.ndarray):
    """Tính toán đầy đủ 4 chỉ số F1 bao gồm cả Scorer chính thức SemEval 2027 (15 lớp)."""
    # 1. Metrics trên 14 frames thuần túy
    frames14_micro = f1_score(y_true_binary, y_pred_binary, average="micro", zero_division=0)
    frames14_macro = f1_score(y_true_binary, y_pred_binary, average="macro", zero_division=0)

    # 2. Scorer chính thức SemEval 2027: Bổ sung cột thứ 15 là 'None'
    # 'None' = 1 khi tất cả 14 nhãn đều bằng 0
    true_has_no_frame = (y_true_binary.sum(axis=1) == 0).astype(int)[:, np.newaxis]
    pred_has_no_frame = (y_pred_binary.sum(axis=1) == 0).astype(int)[:, np.newaxis]

    y_true_15 = np.hstack([y_true_binary, true_has_no_frame])
    y_pred_15 = np.hstack([y_pred_binary, pred_has_no_frame])

    official_micro = f1_score(y_true_15, y_pred_15, average="micro", zero_division=1)
    official_macro = f1_score(y_true_15, y_pred_15, average="macro", zero_division=1)

    return {
        "official_micro_f1": float(official_micro),
        "official_macro_f1": float(official_macro),
        "frames14_micro_f1": float(frames14_micro),
        "frames14_macro_f1": float(frames14_macro),
    }

def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    probabilities = torch.sigmoid(torch.tensor(logits)).numpy()
    y_pred = (probabilities >= INFERENCE_THRESHOLD).astype(int)
    y_true = eval_pred.label_ids.astype(int)

    metrics = compute_official_semeval_metrics(y_true, y_pred)
    # Trả về cả tên rút gọn để Trainer theo dõi tiện lợi
    metrics["macro_f1"] = metrics["frames14_macro_f1"]
    metrics["micro_f1"] = metrics["frames14_micro_f1"]
    return metrics


## 11. Khởi tạo Mô hình Backbone Large & Huấn luyện


In [11]:
id2label = {i: label for i, label in enumerate(LABEL_COLUMNS)}
label2id = {label: i for i, label in id2label.items()}

# Khởi tạo mô hình phân loại đa nhãn 14 classes trên Backbone Cỡ lớn
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Kích hoạt Gradient Checkpointing để tiết kiệm VRAM khi dùng Large Model
if torch.cuda.is_available():
    model.gradient_checkpointing_enable()

supports_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
supports_fp16 = torch.cuda.is_available() and not supports_bf16

# Kiểm tra tương thích eval_strategy / evaluation_strategy theo version Transformers
import inspect
init_params = inspect.signature(TrainingArguments.__init__).parameters
eval_strategy_key = "eval_strategy" if "eval_strategy" in init_params else "evaluation_strategy"

train_args_kwargs = {
    "output_dir": "./checkpoints_exp_large",
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_RATIO,
    eval_strategy_key: "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "official_micro_f1",
    "greater_is_better": True,
    "save_total_limit": 1,
    "bf16": supports_bf16,
    "fp16": supports_fp16,
    "logging_steps": 50,
    "report_to": "none",
    "seed": SEED,
    "dataloader_pin_memory": torch.cuda.is_available(),
}

training_args = TrainingArguments(**train_args_kwargs)

trainer = WeightedMultilabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)],
    pos_weight=pos_weight,
)

print(f"Bắt đầu huấn luyện {MODEL_NAME}...")
print(f"  • Token budget: MAX={MAX_LENGTH} | Title={TITLE_MAX_TOKENS} | Para={PARAGRAPH_MAX_TOKENS}")
print(f"  • Precision: bf16={supports_bf16}, fp16={supports_fp16}")

trainer.train()

# Lưu checkpoint tốt nhất và tokenizer
trainer.save_model(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))

metadata = {
    "base_model": MODEL_NAME,
    "is_large": IS_LARGE_MODEL,
    "protocol": EXPERIMENT_PROTOCOL,
    "directions": ["H03_large_backbone", "H04_context_enriched", "H05_group_split", "H01_per_class_thresholds"],
    "label_columns": LABEL_COLUMNS,
    "max_length": MAX_LENGTH,
    "paragraph_max_tokens": PARAGRAPH_MAX_TOKENS,
    "title_max_tokens": TITLE_MAX_TOKENS,
    "split_strategy": "StratifiedGroupSplit_by_doc_id",
    "best_metric": float(trainer.state.best_metric) if trainer.state.best_metric is not None else None,
}
with open(MODEL_SAVE_DIR / "metadata_exp_large.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\n[Thành công] Đã lưu mô hình tốt nhất vào: {MODEL_SAVE_DIR.resolve()}")


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /kaggle/input/models/aisuko/xlm-roberta-large/transformers/default/1
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Bắt đầu huấn luyện /kaggle/input/models/aisuko/xlm-roberta-large/transformers/default/1...
  • Token budget: MAX=306 | Title=48 | Para=254
  • Precision: bf16=True, fp16=False


Epoch,Training Loss,Validation Loss,Official Micro F1,Official Macro F1,Frames14 Micro F1,Frames14 Macro F1,Macro F1,Micro F1
1,1.289853,0.604004,0.140881,0.056140,0.205505,0.060150,0.060150,0.205505
2,0.938314,0.446388,0.471861,0.361052,0.488789,0.386841,0.386841,0.488789
3,0.840722,0.418905,0.545813,0.465336,0.550149,0.498574,0.498574,0.550149
4,0.699490,0.402504,0.560901,0.460508,0.563207,0.493402,0.493402,0.563207
5,0.578342,0.390383,0.563353,0.479143,0.564453,0.513368,0.513368,0.564453
6,0.475938,0.406799,0.537849,0.468036,0.539461,0.501467,0.501467,0.539461
7,0.400722,0.416620,0.569697,0.484361,0.570273,0.518959,0.518959,0.570273
8,0.331006,0.422111,0.572961,0.481454,0.574812,0.515844,0.515844,0.574812
9,0.291367,0.438325,0.565172,0.469305,0.565762,0.502827,0.502827,0.565762
10,0.230193,0.452972,0.557244,0.470778,0.557809,0.504405,0.504405,0.557809


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[Thành công] Đã lưu mô hình tốt nhất vào: /kaggle/working/model_large


## 12. Đánh giá Chi tiết trên Tập Validation (Ngưỡng Mặc định 0.50)


In [12]:
# Dự đoán xác suất trên tập Validation
val_predictions = trainer.predict(val_dataset)
val_logits = val_predictions.predictions[0] if isinstance(val_predictions.predictions, tuple) else val_predictions.predictions
val_probs = torch.sigmoid(torch.tensor(val_logits)).numpy()
val_targets = val_predictions.label_ids.astype(int)

# Đánh giá tại ngưỡng mặc định 0.50
val_preds_default = (val_probs >= INFERENCE_THRESHOLD).astype(int)
default_metrics = compute_official_semeval_metrics(val_targets, val_preds_default)

print("=== BÁO CÁO PHÂN LOẠI CHI TIẾT TRÊN TẬP VALIDATION (NGƯỠNG 0.50) ===")
print(classification_report(
    val_targets,
    val_preds_default,
    target_names=LABEL_COLUMNS,
    digits=4,
    zero_division=0
))

print(f"--- TỔNG KẾT VALIDATION (NGƯỠNG 0.50) ---")
print(f"  • Official Micro-F1 (15 classes) : {default_metrics['official_micro_f1']:.4f}")
print(f"  • Official Macro-F1 (15 classes) : {default_metrics['official_macro_f1']:.4f}")
print(f"  • Frames14 Micro-F1 (14 classes) : {default_metrics['frames14_micro_f1']:.4f}")
print(f"  • Frames14 Macro-F1 (14 classes) : {default_metrics['frames14_macro_f1']:.4f}")


=== BÁO CÁO PHÂN LOẠI CHI TIẾT TRÊN TẬP VALIDATION (NGƯỠNG 0.50) ===
                                              precision    recall  f1-score   support

                      Capacity_and_resources     0.6129    0.6333    0.6230        30
                        Crime_and_punishment     0.5517    0.6667    0.6038        24
                           Cultural_identity     0.5000    0.4286    0.4615        14
                                    Economic     0.6207    0.7500    0.6792        48
          External_regulation_and_reputation     0.4839    0.4286    0.4545        35
                       Fairness_and_equality     0.4286    0.1765    0.2500        17
                           Health_and_safety     0.5439    0.5962    0.5688        52
Legality_Constitutionality_and_jurisprudence     0.4286    0.5806    0.4932        31
                                    Morality     0.5833    0.5833    0.5833        24
          Policy_prescription_and_evaluation     0.2963    0.3077    0

## 13. Hướng 01: Dò Tìm Ngưỡng Quyết định Tối ưu (Per-Class Thresholds - Coordinate Search)

Bằng cách áp dụng **Coordinate Search** trên tập Validation:
1. **Bước 1:** Quét tìm ngưỡng toàn cục tốt nhất $\theta_0 \in [0.20, 0.70]$.
2. **Bước 2:** Dò tìm ngưỡng độc lập tối ưu $\vec{\theta} = [\theta_1, \dots, \theta_{14}]$ cho từng nhãn (2 vòng lặp).
3. **Bước 3:** Đối chiếu với **Bảng Ngưỡng Khóa Chuẩn** (`benchmark_lock.json`).


In [13]:
# 1. Quét tìm ngưỡng toàn cục tối ưu (Global Grid Search)
print("=== BƯỚC 1: QUÉT NGƯỠNG TOÀN CỤC TRÊN VALIDATION ===")
global_thresholds = np.arange(0.20, 0.70, 0.05)
best_global_th = INFERENCE_THRESHOLD
best_global_score = 0.0
grid_rows = []

for th in global_thresholds:
    p_bin = (val_probs >= th).astype(int)
    m = compute_official_semeval_metrics(val_targets, p_bin)
    grid_rows.append({
        "Threshold": round(th, 2),
        "Official Micro-F1": m["official_micro_f1"],
        "Official Macro-F1": m["official_macro_f1"],
        "Frames14 Macro-F1": m["frames14_macro_f1"]
    })
    if m["official_micro_f1"] > best_global_score:
        best_global_score = m["official_micro_f1"]
        best_global_th = th

display(pd.DataFrame(grid_rows))
print(f"=> Ngưỡng toàn cục tốt nhất: {best_global_th:.2f} (Official Micro-F1 = {best_global_score:.4f})")

# 2. Coordinate Search tìm 14 ngưỡng độc lập (Per-Class Optimization) - Hướng 01
print("\n=== BƯỚC 2: COORDINATE SEARCH 14 NGƯỠNG ĐỘC LẬP (2 VÒNG LẶP) ===")
current_thresholds = np.full(NUM_LABELS, best_global_th)
search_grid = np.arange(0.15, 0.85, 0.05)

for iteration in range(2):
    print(f"Vòng lặp {iteration + 1}/2:")
    for col_idx, col_name in enumerate(LABEL_COLUMNS):
        best_th_col = current_thresholds[col_idx]
        best_f1_col = -1.0

        for candidate_th in search_grid:
            temp_thresholds = current_thresholds.copy()
            temp_thresholds[col_idx] = candidate_th

            temp_preds = (val_probs >= temp_thresholds).astype(int)
            temp_f1 = f1_score(val_targets[:, col_idx], temp_preds[:, col_idx], zero_division=0)

            if temp_f1 > best_f1_col:
                best_f1_col = temp_f1
                best_th_col = candidate_th

        current_thresholds[col_idx] = best_th_col

# Hiển thị bảng ngưỡng đã tối ưu
coord_th_df = pd.DataFrame({
    "Nhãn": LABEL_COLUMNS,
    "Ngưỡng Coordinate Search": [round(t, 2) for t in current_thresholds]
})
print("\nBảng 14 ngưỡng tối ưu sau Coordinate Search:")
display(coord_th_df)

# 3. Bảng Ngưỡng Khóa Thực nghiệm từ bs2023AB_da_train (benchmark_lock.json)
LOCKED_BENCHMARK_THRESHOLDS = {
    "Health_and_safety": 0.20,
    "Economic": 0.25,
    "External_regulation_and_reputation": 0.35,
    "Security_and_defense": 0.40,
    "Crime_and_punishment": 0.45,
    "Political": 0.45,
    "Quality_of_life": 0.50,
    "Capacity_and_resources": 0.55,
    "Cultural_identity": 0.60,
    "Legality_Constitutionality_and_jurisprudence": 0.60,
    "Fairness_and_equality": 0.65,
    "Morality": 0.70,
    "Policy_prescription_and_evaluation": 0.70,
    "Public_opinion": 0.70
}
locked_th_vector = np.array([LOCKED_BENCHMARK_THRESHOLDS[col] for col in LABEL_COLUMNS])

# 4. Đánh giá so sánh 4 bộ ngưỡng trên tập Validation
preds_global = (val_probs >= best_global_th).astype(int)
preds_coord = (val_probs >= current_thresholds).astype(int)
preds_locked = (val_probs >= locked_th_vector).astype(int)

comp_df = pd.DataFrame([
    {"Phương án": "Ngưỡng mặc định 0.50", **compute_official_semeval_metrics(val_targets, val_preds_default)},
    {"Phương án": f"Ngưỡng toàn cục ({best_global_th:.2f})", **compute_official_semeval_metrics(val_targets, preds_global)},
    {"Phương án": "Ngưỡng Coordinate Search mới", **compute_official_semeval_metrics(val_targets, preds_coord)},
    {"Phương án": "Ngưỡng Khóa Benchmark Lock", **compute_official_semeval_metrics(val_targets, preds_locked)},
])
print("\n=== BẢNG SO SÁNH 4 BỘ NGƯỠNG TRÊN VALIDATION ===")
display(comp_df)

# Lưu vector ngưỡng tối ưu vào JSON
threshold_dict = {col: float(current_thresholds[i]) for i, col in enumerate(LABEL_COLUMNS)}
with open(MODEL_SAVE_DIR / "optimal_thresholds_large.json", "w", encoding="utf-8") as f:
    json.dump(threshold_dict, f, indent=2)
print(f"\n[Thành công] Đã lưu 14 ngưỡng tối ưu vào: {MODEL_SAVE_DIR / 'optimal_thresholds_large.json'}")


=== BƯỚC 1: QUÉT NGƯỠNG TOÀN CỤC TRÊN VALIDATION ===


,Threshold,Official Micro-F1,Official Macro-F1,Frames14 Macro-F1
0,0.20,0.544529,0.533769,0.500467
1,0.25,0.551351,0.538410,0.505439
2,0.30,0.565258,0.553866,0.521999
3,0.35,0.580078,0.499897,0.535604
4,0.40,0.584830,0.504310,0.540332
5,0.45,0.577640,0.483944,0.518512
6,0.50,0.572961,0.481454,0.515844
7,0.55,0.564835,0.468401,0.501858
8,0.60,0.565315,0.473056,0.506846
9,0.65,0.572414,0.479596,0.513853


=> Ngưỡng toàn cục tốt nhất: 0.40 (Official Micro-F1 = 0.5848)

=== BƯỚC 2: COORDINATE SEARCH 14 NGƯỠNG ĐỘC LẬP (2 VÒNG LẶP) ===
Vòng lặp 1/2:
Vòng lặp 2/2:

Bảng 14 ngưỡng tối ưu sau Coordinate Search:


,Nhãn,Ngưỡng Coordinate Search
0,Capacity_and_resources,0.35
1,Crime_and_punishment,0.45
2,Cultural_identity,0.65
3,Economic,0.65
4,External_regulation_and_reputation,0.70
5,Fairness_and_equality,0.15
6,Health_and_safety,0.40
7,Legality_Constitutionality_and_jurisprudence,0.80
8,Morality,0.50
9,Policy_prescription_and_evaluation,0.75



=== BẢNG SO SÁNH 4 BỘ NGƯỠNG TRÊN VALIDATION ===


,Phương án,official_micro_f1,official_macro_f1,frames14_micro_f1,frames14_macro_f1
0,Ngưỡng mặc định 0.50,0.572961,0.481454,0.574812,0.515844
1,Ngưỡng toàn cục (0.40),0.584830,0.504310,0.586000,0.540332
2,Ngưỡng Coordinate Search mới,0.612335,0.531639,0.615044,0.569613
3,Ngưỡng Khóa Benchmark Lock,0.579832,0.478720,0.581053,0.512914



[Thành công] Đã lưu 14 ngưỡng tối ưu vào: /kaggle/working/model_large/optimal_thresholds_large.json


## 14. Kiểm nghiệm Mù trên Tập Test B (Protocol 1 Benchmark A → B)

> [!IMPORTANT]
> **MỐC ĐỐI CHỨNG CẦN ĐẠT ĐƯỢC TRÊN TẬP TEST B (1.768 đoạn, 9 ngôn ngữ):**
> * **Official Micro-F1 : `0.4564`**
> * **Official Macro-F1 : `0.3969`**
> * **Frames14 Macro-F1 : `0.4253`**

Đánh giá mù trên tập Test B bằng cả 2 bộ ngưỡng: **Coordinate Search** và **Locked Benchmark**.


In [14]:
if has_test_b and len(test_raw_df) > 0:
    print("=== ĐÁNH GIÁ MÙ TRÊN TẬP TEST B (1.768 ĐOẠN, 9 NGÔN NGỮ) ===")
    test_dataset = AuditedContextParagraphDataset(test_raw_df, tokenizer, LABEL_COLUMNS, is_test=False)

    test_predictions = trainer.predict(test_dataset)
    test_logits = test_predictions.predictions[0] if isinstance(test_predictions.predictions, tuple) else test_predictions.predictions
    test_probs = torch.sigmoid(torch.tensor(test_logits)).numpy()
    test_targets = test_predictions.label_ids.astype(int)

    # Dự đoán bằng bộ ngưỡng Coordinate Search và Ngưỡng Khóa
    test_preds_coord = (test_probs >= current_thresholds).astype(int)
    test_preds_locked = (test_probs >= locked_th_vector).astype(int)
    test_preds_default = (test_probs >= INFERENCE_THRESHOLD).astype(int)

    m_default = compute_official_semeval_metrics(test_targets, test_preds_default)
    m_coord = compute_official_semeval_metrics(test_targets, test_preds_coord)
    m_locked = compute_official_semeval_metrics(test_targets, test_preds_locked)

    print(f"\nKẾT QUẢ ĐỐI CHỨNG TRÊN TEST B:")
    testb_comp = pd.DataFrame([
        {"Phương án": "Ngưỡng mặc định 0.50", **m_default},
        {"Phương án": "Coordinate Search Thresholds", **m_coord},
        {"Phương án": "Locked Benchmark Thresholds", **m_locked},
        {"Phương án": "--- MỐC CHUẨN bs2023AB ---", "official_micro_f1": 0.4564, "official_macro_f1": 0.3969, "frames14_micro_f1": None, "frames14_macro_f1": 0.4253},
    ])
    display(testb_comp)

    # Phân tích theo từng ngôn ngữ trên Test B (dùng bộ ngưỡng Locked)
    test_raw_df_copy = test_raw_df.copy()
    test_raw_df_copy["_pred_bin"] = list(test_preds_locked)
    test_raw_df_copy["_target_bin"] = list(test_targets)

    lang_results = []
    seen_languages = {"DE", "FR", "IT", "PL", "RU"}
    for lang, group in test_raw_df_copy.groupby("language"):
        y_t = np.vstack(group["_target_bin"].values)
        y_p = np.vstack(group["_pred_bin"].values)
        lm = compute_official_semeval_metrics(y_t, y_p)
        lang_results.append({
            "Ngôn ngữ": lang,
            "Số đoạn": len(group),
            "Tập Train A": "Seen" if lang in seen_languages else "Unseen (Zero-shot)",
            "Official Micro-F1": lm["official_micro_f1"],
            "Frames14 Macro-F1": lm["frames14_macro_f1"]
        })

    df_lang = pd.DataFrame(lang_results).sort_values("Official Micro-F1", ascending=False)
    print("\nHiệu năng chi tiết theo ngôn ngữ trên Test B:")
    display(df_lang)
else:
    print("[Thông báo] Bỏ qua kiểm nghiệm Test B (Đang chạy ở chế độ Protocol 2 Full Dataset Training).")


=== ĐÁNH GIÁ MÙ TRÊN TẬP TEST B (1.768 ĐOẠN, 9 NGÔN NGỮ) ===



KẾT QUẢ ĐỐI CHỨNG TRÊN TEST B:


,Phương án,official_micro_f1,official_macro_f1,frames14_micro_f1,frames14_macro_f1
0,Ngưỡng mặc định 0.50,0.489453,0.432912,0.493062,0.463835
1,Coordinate Search Thresholds,0.487612,0.426148,0.494112,0.456587
2,Locked Benchmark Thresholds,0.489625,0.424357,0.493973,0.454668
3,--- MỐC CHUẨN bs2023AB ---,0.456400,0.396900,NaN,0.425300



Hiệu năng chi tiết theo ngôn ngữ trên Test B:


,Ngôn ngữ,Số đoạn,Tập Train A,Official Micro-F1,Frames14 Macro-F1
6,KA,56,Unseen (Zero-shot),0.613139,0.406312
4,GR,368,Unseen (Zero-shot),0.537313,0.439531
5,IT,188,Seen,0.495512,0.437790
0,DE,316,Seen,0.490872,0.457429
2,ES,61,Unseen (Zero-shot),0.485207,0.401772
7,PL,264,Seen,0.470893,0.458064
1,EN,220,Unseen (Zero-shot),0.453925,0.431097
8,RU,87,Seen,0.444444,0.352886
3,FR,160,Seen,0.440281,0.431490


## 15. Thử nghiệm Dự đoán Văn bản mới kèm Tiêu đề


In [15]:
def predict_paragraph_with_context(paragraph: str, title: str = "", thresholds: dict = None):
    """Dự đoán nhãn framing cho một đoạn văn bản mới có bổ sung tiêu đề bài báo."""
    if thresholds is None:
        thresholds = LOCKED_BENCHMARK_THRESHOLDS

    p_ids = tokenizer.encode(paragraph, add_special_tokens=False, truncation=True, max_length=PARAGRAPH_MAX_TOKENS)
    t_ids = tokenizer.encode(title, add_special_tokens=False, truncation=True, max_length=TITLE_MAX_TOKENS) if title else []

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    if t_ids:
        input_ids = [cls_id] + t_ids + [sep_id, sep_id] + p_ids + [sep_id]
    else:
        input_ids = [cls_id] + p_ids + [sep_id]

    inputs = {
        "input_ids": torch.tensor([input_ids], device=device),
        "attention_mask": torch.tensor([[1] * len(input_ids)], device=device),
    }

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits)[0].cpu().numpy()

    predicted_frames = []
    print(f"Tiêu đề bài báo : {title if title else '(Không có)'}")
    print(f"Đoạn văn bản   : {paragraph}")
    for idx, col in enumerate(LABEL_COLUMNS):
        th = thresholds.get(col, 0.50)
        if probs[idx] >= th:
            predicted_frames.append(col)

    print(f"Dự đoán Framing : {predicted_frames}")
    print(f"\nĐiểm xác suất chi tiết:")
    sorted_probs = sorted(zip(LABEL_COLUMNS, probs), key=lambda x: x[1], reverse=True)
    for col, p in sorted_probs:
        bar = "█" * int(p * 20)
        th = thresholds.get(col, 0.50)
        mark = "✓" if p >= th else " "
        print(f"  [{mark}] {col:45s}: {p:.4f} (th={th:.2f}) {bar}")

# Ví dụ thử nghiệm thực tế
predict_paragraph_with_context(
    paragraph="Lawmakers approved 50 billion euros to modernize naval infrastructure and reinforce borders.",
    title="EU announces emergency economic package to address escalating defense spending"
)


Tiêu đề bài báo : EU announces emergency economic package to address escalating defense spending
Đoạn văn bản   : Lawmakers approved 50 billion euros to modernize naval infrastructure and reinforce borders.
Dự đoán Framing : ['Economic', 'Security_and_defense']

Điểm xác suất chi tiết:
  [✓] Security_and_defense                         : 0.9466 (th=0.40) ██████████████████
  [✓] Economic                                     : 0.5166 (th=0.25) ██████████
  [ ] Policy_prescription_and_evaluation           : 0.2379 (th=0.70) ████
  [ ] Capacity_and_resources                       : 0.1883 (th=0.55) ███
  [ ] External_regulation_and_reputation           : 0.0550 (th=0.35) █
  [ ] Legality_Constitutionality_and_jurisprudence : 0.0427 (th=0.60) 
  [ ] Crime_and_punishment                         : 0.0379 (th=0.45) 
  [ ] Political                                    : 0.0357 (th=0.45) 
  [ ] Cultural_identity                            : 0.0233 (th=0.60) 
  [ ] Health_and_safety               

## 16. Xuất File Kết quả Dự đoán Chuẩn SemEval Task 9 (TSV Format)


In [16]:
def export_semeval_predictions(dataframe: pd.DataFrame, model, tokenizer, output_file: Path, thresholds: dict = None):
    """Xuất file dự đoán chuẩn định dạng SemEval TSV."""
    if thresholds is None:
        thresholds = LOCKED_BENCHMARK_THRESHOLDS

    dataset = AuditedContextParagraphDataset(dataframe, tokenizer, LABEL_COLUMNS, is_test=True)
    eval_trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="./_tmp_eval",
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            report_to="none"
        ),
        data_collator=DynamicPaddingDataCollator(tokenizer)
    )

    pred_res = eval_trainer.predict(dataset)
    logits = pred_res.predictions[0] if isinstance(pred_res.predictions, tuple) else pred_res.predictions
    probs = torch.sigmoid(torch.tensor(logits)).numpy()

    th_vector = np.array([thresholds.get(col, 0.50) for col in LABEL_COLUMNS])
    binary_preds = (probs >= th_vector).astype(int)

    lines = []
    for idx, row in dataframe.iterrows():
        doc_id = str(row.get("doc_id", f"sample_{idx}"))
        begin_span = str(row.get("begin_span", "0"))
        end_span = str(row.get("end_span", "0"))

        assigned_frames = [LABEL_COLUMNS[i] for i, val in enumerate(binary_preds[idx]) if val == 1]

        if assigned_frames:
            line = f"{doc_id}\t{begin_span}\t{end_span}\t" + "\t".join(assigned_frames)
        else:
            line = f"{doc_id}\t{begin_span}\t{end_span}"
        lines.append(line)

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

    print(f"[Thành công] Đã xuất {len(lines):,} dòng dự đoán vào: {output_file.resolve()}")

# Xuất file dự đoán trên tập Validation
val_export_coord = OUTPUT_DIR / "subtask-1-exp-large-val-coord-predictions.tsv"
export_semeval_predictions(val_df, model, tokenizer, val_export_coord, thresholds=threshold_dict)

val_export_locked = OUTPUT_DIR / "subtask-1-exp-large-val-locked-predictions.tsv"
export_semeval_predictions(val_df, model, tokenizer, val_export_locked, thresholds=LOCKED_BENCHMARK_THRESHOLDS)

# Xuất file dự đoán trên tập Test B nếu có
if has_test_b and len(test_raw_df) > 0:
    test_export_coord = OUTPUT_DIR / "subtask-1-exp-large-testB-coord-predictions.tsv"
    export_semeval_predictions(test_raw_df, model, tokenizer, test_export_coord, thresholds=threshold_dict)

    test_export_locked = OUTPUT_DIR / "subtask-1-exp-large-testB-locked-predictions.tsv"
    export_semeval_predictions(test_raw_df, model, tokenizer, test_export_locked, thresholds=LOCKED_BENCHMARK_THRESHOLDS)


[Thành công] Đã xuất 346 dòng dự đoán vào: /kaggle/working/output/subtask-1-exp-large-val-coord-predictions.tsv


[Thành công] Đã xuất 346 dòng dự đoán vào: /kaggle/working/output/subtask-1-exp-large-val-locked-predictions.tsv


[Thành công] Đã xuất 1,720 dòng dự đoán vào: /kaggle/working/output/subtask-1-exp-large-testB-coord-predictions.tsv


[Thành công] Đã xuất 1,720 dòng dự đoán vào: /kaggle/working/output/subtask-1-exp-large-testB-locked-predictions.tsv


In [17]:
import shutil

# Xóa thư mục tạm thời nếu cần
shutil.rmtree("./_tmp_eval", ignore_errors=True)
print("Hoàn tất quy trình huấn luyện và kiểm nghiệm mô hình Large!")


Hoàn tất quy trình huấn luyện và kiểm nghiệm mô hình Large!
